## Setup Project dan Package

In [1]:
from pathlib import Path
from datetime import datetime
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import GroupShuffleSplit

try:
    from xgboost import XGBClassifier
    xgboost_tersedia = True
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier
    xgboost_tersedia = True

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_multi = direktori_project / "data" / "processed" / "multi_dataset"
direktori_models = direktori_project / "models"
direktori_outputs = direktori_project / "reports" / "outputs"

direktori_models.mkdir(parents=True, exist_ok=True)
direktori_outputs.mkdir(parents=True, exist_ok=True)

lokasi_dataset_v5 = direktori_multi / "dataset_training_multi_v5.csv"
lokasi_fitur_v5 = direktori_multi / "fitur_training_multi_v5.csv"
lokasi_target_v5 = direktori_multi / "target_training_multi_v5.csv"
lokasi_daftar_fitur_v5 = direktori_outputs / "daftar_fitur_multi_dataset_v5.json"

print("Direktori project:", direktori_project)
print("Dataset V5:", lokasi_dataset_v5.exists())
print("Fitur V5:", lokasi_fitur_v5.exists())
print("Target V5:", lokasi_target_v5.exists())
print("Daftar fitur V5:", lokasi_daftar_fitur_v5.exists())

Direktori project: C:\Users\ASUS\PHISHING
Dataset V5: True
Fitur V5: True
Target V5: True
Daftar fitur V5: True


## Load Dataset Training

In [2]:
with open(lokasi_daftar_fitur_v5, "r", encoding="utf-8") as file:
    daftar_fitur_v5 = json.load(file)

metadata_cols = [
    "url",
    "domain",
    "original_label",
    "target_phishing",
    "dataset_name",
    "sumber_data",
    "split",
]

metadata_v5 = pd.read_csv(
    lokasi_dataset_v5,
    usecols=metadata_cols,
    dtype=str,
    low_memory=False,
)

data_target = pd.read_csv(
    lokasi_target_v5,
    low_memory=False,
)

data_fitur = pd.read_csv(
    lokasi_fitur_v5,
    low_memory=False,
)

for kolom in daftar_fitur_v5:
    if kolom not in data_fitur.columns:
        data_fitur[kolom] = 0

X = data_fitur[daftar_fitur_v5].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)

y = data_target["target_phishing"].astype(int)
metadata_v5["target_phishing"] = metadata_v5["target_phishing"].astype(int)

if len(X) != len(y) or len(X) != len(metadata_v5):
    raise ValueError("Jumlah baris fitur, target, dan metadata tidak sama.")

print("Data training V5 berhasil dibaca.")
print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)

display(metadata_v5["dataset_name"].value_counts())
display(y.value_counts())

Data training V5 berhasil dibaca.
Ukuran X: (975090, 49)
Ukuran y: (975090,)


dataset_name
DeepURLBench    359819
PhreshPhish     279936
PhiUSIIL        235365
Tranco           99970
Name: count, dtype: int64

target_phishing
0    509023
1    466067
Name: count, dtype: int64

## Split Train/Test Berdasarkan Domain

In [3]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

groups = metadata_v5["domain"].fillna(metadata_v5["url"]).astype(str)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

metadata_train = metadata_v5.iloc[train_idx].reset_index(drop=True)
metadata_test = metadata_v5.iloc[test_idx].reset_index(drop=True)

metadata_split = metadata_v5.copy()
metadata_split["split_model_v5"] = "train"
metadata_split.loc[test_idx, "split_model_v5"] = "test"

lokasi_split_v5 = direktori_outputs / "split_model_multi_dataset_v5.csv"
metadata_split.to_csv(lokasi_split_v5, index=False, encoding="utf-8")

print("Split train/test selesai.")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

display(y_train.value_counts())
display(y_test.value_counts())
display(metadata_test["dataset_name"].value_counts())

Split train/test selesai.
X_train: (776771, 49)
X_test: (198319, 49)


target_phishing
0    408143
1    368628
Name: count, dtype: int64

target_phishing
0    100880
1     97439
Name: count, dtype: int64

dataset_name
DeepURLBench    71679
PhreshPhish     59173
PhiUSIIL        47339
Tranco          20128
Name: count, dtype: int64

## Buat Sample Weight

In [4]:
def buat_sample_weight(y_data, metadata_data):
    jumlah_data = len(y_data)

    class_counts = y_data.value_counts().to_dict()
    dataset_counts = metadata_data["dataset_name"].value_counts().to_dict()

    class_weight = {
        kelas: jumlah_data / (len(class_counts) * jumlah)
        for kelas, jumlah in class_counts.items()
    }

    dataset_weight = {
        dataset: jumlah_data / (len(dataset_counts) * jumlah)
        for dataset, jumlah in dataset_counts.items()
    }

    bobot = (
        y_data.map(class_weight).astype(float).values
        * metadata_data["dataset_name"].map(dataset_weight).astype(float).values
    )

    bobot = bobot / np.mean(bobot)

    return bobot.astype(np.float32)


sample_weight_train = buat_sample_weight(y_train, metadata_train)

print("Sample weight siap.")
print("Rata-rata:", round(float(np.mean(sample_weight_train)), 4))
print("Minimum:", round(float(np.min(sample_weight_train)), 4))
print("Maksimum:", round(float(np.max(sample_weight_train)), 4))

Sample weight siap.
Rata-rata: 1.0
Minimum: 0.6474
Maksimum: 2.3364


## Training Random Forest

In [5]:
rf_v5 = RandomForestClassifier(
    n_estimators=220,
    max_depth=24,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)

rf_v5.fit(
    X_train,
    y_train,
    sample_weight=sample_weight_train,
)

lokasi_model_rf_v5 = direktori_models / "model_rf_multi_dataset_v5.pkl"
joblib.dump(rf_v5, lokasi_model_rf_v5)

print("Random Forest V5 selesai dilatih.")
print("Model RF V5:", lokasi_model_rf_v5)

Random Forest V5 selesai dilatih.
Model RF V5: C:\Users\ASUS\PHISHING\models\model_rf_multi_dataset_v5.pkl


## Training XGBoost

In [6]:
xgb_v5 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgb_v5.fit(
    X_train,
    y_train,
    sample_weight=sample_weight_train,
)

lokasi_model_xgb_v5 = direktori_models / "model_xgb_multi_dataset_v5.pkl"
joblib.dump(xgb_v5, lokasi_model_xgb_v5)

print("XGBoost V5 selesai dilatih.")
print("Model XGBoost V5:", lokasi_model_xgb_v5)

XGBoost V5 selesai dilatih.
Model XGBoost V5: C:\Users\ASUS\PHISHING\models\model_xgb_multi_dataset_v5.pkl


## Evaluasi Model

In [7]:
def ambil_probabilitas(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]

    prediksi = model.predict(X_data)
    return np.array(prediksi, dtype=float)


def evaluasi_model(nama_model, model, X_data, y_data, threshold=0.5):
    probabilitas = ambil_probabilitas(model, X_data)
    prediksi = (probabilitas >= threshold).astype(int)

    hasil = {
        "nama_model": nama_model,
        "threshold": threshold,
        "accuracy": accuracy_score(y_data, prediksi),
        "precision_berisiko": precision_score(y_data, prediksi, zero_division=0),
        "recall_berisiko": recall_score(y_data, prediksi, zero_division=0),
        "f1_berisiko": f1_score(y_data, prediksi, zero_division=0),
        "roc_auc": roc_auc_score(y_data, probabilitas),
    }

    return hasil, prediksi, probabilitas


hasil_rf, pred_rf, prob_rf = evaluasi_model(
    "Random Forest Multi Dataset V5",
    rf_v5,
    X_test,
    y_test,
)

hasil_xgb, pred_xgb, prob_xgb = evaluasi_model(
    "XGBoost Multi Dataset V5",
    xgb_v5,
    X_test,
    y_test,
)

data_evaluasi_v5 = pd.DataFrame([hasil_rf, hasil_xgb])
data_evaluasi_v5 = data_evaluasi_v5.sort_values(
    ["roc_auc", "f1_berisiko", "recall_berisiko"],
    ascending=False,
).reset_index(drop=True)

lokasi_evaluasi_v5 = direktori_outputs / "hasil_evaluasi_model_multi_dataset_v5.csv"
data_evaluasi_v5.to_csv(lokasi_evaluasi_v5, index=False, encoding="utf-8")

print("Evaluasi model selesai.")
display(data_evaluasi_v5)

model_terbaik_nama = data_evaluasi_v5.loc[0, "nama_model"]

if "Random Forest" in model_terbaik_nama:
    model_terbaik_v5 = rf_v5
    pred_terbaik = pred_rf
    prob_terbaik = prob_rf
else:
    model_terbaik_v5 = xgb_v5
    pred_terbaik = pred_xgb
    prob_terbaik = prob_xgb

lokasi_model_terbaik_v5 = direktori_models / "model_terbaik_multi_dataset_v5.pkl"
joblib.dump(model_terbaik_v5, lokasi_model_terbaik_v5)

print("Model terbaik:", model_terbaik_nama)
print("File model terbaik:", lokasi_model_terbaik_v5)

Evaluasi model selesai.


,nama_model,threshold,accuracy,precision_berisiko,recall_berisiko,f1_berisiko,roc_auc
0,Random Forest Multi Dataset V5,0.5,0.885780,0.903047,0.859841,0.880914,0.954131
1,XGBoost Multi Dataset V5,0.5,0.878761,0.893526,0.855140,0.873912,0.946945


Model terbaik: Random Forest Multi Dataset V5
File model terbaik: C:\Users\ASUS\PHISHING\models\model_terbaik_multi_dataset_v5.pkl


## Classification Report dan Confusion Matrix

In [8]:
laporan_terbaik = classification_report(
    y_test,
    pred_terbaik,
    target_names=["Aman", "Berisiko"],
    output_dict=True,
    zero_division=0,
)

data_laporan_terbaik = pd.DataFrame(laporan_terbaik).T

cm = confusion_matrix(y_test, pred_terbaik)
data_confusion = pd.DataFrame(
    cm,
    index=["Aktual_Aman", "Aktual_Berisiko"],
    columns=["Prediksi_Aman", "Prediksi_Berisiko"],
)

lokasi_laporan_terbaik = direktori_outputs / "classification_report_model_terbaik_v5.csv"
lokasi_confusion_v5 = direktori_outputs / "confusion_matrix_model_terbaik_v5.csv"

data_laporan_terbaik.to_csv(lokasi_laporan_terbaik, encoding="utf-8")
data_confusion.to_csv(lokasi_confusion_v5, encoding="utf-8")

print("Classification report model terbaik:")
display(data_laporan_terbaik)

print("Confusion matrix model terbaik:")
display(data_confusion)

Classification report model terbaik:


,precision,recall,f1-score,support
Aman,0.870601,0.910835,0.890264,100880.00000
Berisiko,0.903047,0.859841,0.880914,97439.00000
accuracy,0.885780,0.885780,0.885780,0.88578
macro avg,0.886824,0.885338,0.885589,198319.00000
weighted avg,0.886543,0.885780,0.885670,198319.00000


Confusion matrix model terbaik:


,Prediksi_Aman,Prediksi_Berisiko
Aktual_Aman,91885,8995
Aktual_Berisiko,13657,83782


## Threshold Tuning Model Terbaik

In [9]:
hasil_threshold = []

for threshold in np.arange(0.10, 0.91, 0.05):
    prediksi_threshold = (prob_terbaik >= threshold).astype(int)

    hasil_threshold.append({
        "threshold": round(float(threshold), 2),
        "precision_berisiko": precision_score(y_test, prediksi_threshold, zero_division=0),
        "recall_berisiko": recall_score(y_test, prediksi_threshold, zero_division=0),
        "f1_berisiko": f1_score(y_test, prediksi_threshold, zero_division=0),
        "accuracy": accuracy_score(y_test, prediksi_threshold),
    })

data_threshold_v5 = pd.DataFrame(hasil_threshold)

kandidat_threshold = data_threshold_v5[
    data_threshold_v5["recall_berisiko"] >= 0.95
].copy()

if kandidat_threshold.empty:
    threshold_terbaik = float(
        data_threshold_v5.sort_values("f1_berisiko", ascending=False).iloc[0]["threshold"]
    )
else:
    threshold_terbaik = float(
        kandidat_threshold.sort_values(
            ["f1_berisiko", "precision_berisiko"],
            ascending=False,
        ).iloc[0]["threshold"]
    )

lokasi_threshold_v5 = direktori_outputs / "threshold_tuning_model_terbaik_v5.csv"
data_threshold_v5.to_csv(lokasi_threshold_v5, index=False, encoding="utf-8")

lokasi_threshold_json_v5 = direktori_outputs / "threshold_model_terbaik_v5.json"
with open(lokasi_threshold_json_v5, "w", encoding="utf-8") as file:
    json.dump(
        {
            "model_terbaik": model_terbaik_nama,
            "threshold_terbaik": threshold_terbaik,
            "catatan": "Threshold dipilih dengan prioritas recall berisiko minimal 0.95 jika tersedia.",
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

print("Threshold tuning selesai.")
print("Threshold terbaik:", threshold_terbaik)
display(data_threshold_v5)

Threshold tuning selesai.
Threshold terbaik: 0.15


,threshold,precision_berisiko,recall_berisiko,f1_berisiko,accuracy
0,0.10,0.720047,0.970618,0.826764,0.800150
1,0.15,0.764466,0.956691,0.849844,0.833899
2,0.20,0.796804,0.943606,0.864014,0.854063
3,0.25,0.820482,0.931742,0.872580,0.866301
4,0.30,0.840105,0.915650,0.876252,0.872932
5,0.35,0.857592,0.902329,0.879391,0.878393
6,0.40,0.876460,0.886975,0.881687,0.883042
7,0.45,0.889681,0.872105,0.880805,0.884030
8,0.50,0.903047,0.859841,0.880914,0.885780
9,0.55,0.916600,0.845832,0.879795,0.886441


## Evaluasi Final dengan Threshold Terbaik

In [10]:
pred_final = (prob_terbaik >= threshold_terbaik).astype(int)

hasil_final_threshold = {
    "nama_model": model_terbaik_nama,
    "threshold": threshold_terbaik,
    "accuracy": accuracy_score(y_test, pred_final),
    "precision_berisiko": precision_score(y_test, pred_final, zero_division=0),
    "recall_berisiko": recall_score(y_test, pred_final, zero_division=0),
    "f1_berisiko": f1_score(y_test, pred_final, zero_division=0),
    "roc_auc": roc_auc_score(y_test, prob_terbaik),
}

data_final_threshold = pd.DataFrame([hasil_final_threshold])

cm_final = confusion_matrix(y_test, pred_final)
data_confusion_final = pd.DataFrame(
    cm_final,
    index=["Aktual_Aman", "Aktual_Berisiko"],
    columns=["Prediksi_Aman", "Prediksi_Berisiko"],
)

lokasi_final_threshold = direktori_outputs / "hasil_final_threshold_model_terbaik_v5.csv"
lokasi_confusion_final = direktori_outputs / "confusion_matrix_threshold_model_terbaik_v5.csv"

data_final_threshold.to_csv(lokasi_final_threshold, index=False, encoding="utf-8")
data_confusion_final.to_csv(lokasi_confusion_final, encoding="utf-8")

print("Evaluasi final threshold selesai.")
display(data_final_threshold)
display(data_confusion_final)

Evaluasi final threshold selesai.


,nama_model,threshold,accuracy,precision_berisiko,recall_berisiko,f1_berisiko,roc_auc
0,Random Forest Multi Dataset V5,0.15,0.833899,0.764466,0.956691,0.849844,0.954131


,Prediksi_Aman,Prediksi_Berisiko
Aktual_Aman,72159,28721
Aktual_Berisiko,4220,93219


## Feature Importance

In [11]:
if hasattr(model_terbaik_v5, "feature_importances_"):
    importance = model_terbaik_v5.feature_importances_
else:
    importance = np.zeros(len(daftar_fitur_v5))

data_importance_v5 = pd.DataFrame({
    "fitur": daftar_fitur_v5,
    "importance": importance,
}).sort_values("importance", ascending=False).reset_index(drop=True)

lokasi_importance_v5 = direktori_outputs / "feature_importance_model_terbaik_v5.csv"
data_importance_v5.to_csv(lokasi_importance_v5, index=False, encoding="utf-8")

print("Feature importance disimpan.")
display(data_importance_v5.head(30))

Feature importance disimpan.


,fitur,importance
0,IsHTTPS,0.201089
1,DomainLength,0.143353
2,NoOfSubDomain,0.083644
3,NoOfOtherSpecialCharsInURL,0.060272
4,uses_digit_substitution,0.054837
5,lookalike_score,0.052073
6,DegitRatioInURL,0.047569
7,NoOfDegitsInURL,0.043616
8,URLLength,0.041533
9,NoOfLettersInURL,0.034271


## Error Analysis

In [12]:
data_error_test = metadata_test.copy()
data_error_test["y_true"] = y_test.values
data_error_test["y_pred"] = pred_final
data_error_test["probabilitas_berisiko"] = prob_terbaik
data_error_test["jenis_error"] = "benar"

data_error_test.loc[
    (data_error_test["y_true"] == 0) & (data_error_test["y_pred"] == 1),
    "jenis_error",
] = "false_positive"

data_error_test.loc[
    (data_error_test["y_true"] == 1) & (data_error_test["y_pred"] == 0),
    "jenis_error",
] = "false_negative"

ringkasan_error = (
    data_error_test
    .groupby(["dataset_name", "jenis_error"])
    .size()
    .reset_index(name="jumlah_data")
    .sort_values(["dataset_name", "jenis_error"])
)

contoh_error = data_error_test[
    data_error_test["jenis_error"] != "benar"
].copy()

contoh_error = contoh_error.sort_values(
    ["jenis_error", "probabilitas_berisiko"],
    ascending=[True, False],
).head(300)

lokasi_ringkasan_error_v5 = direktori_outputs / "ringkasan_error_model_terbaik_v5.csv"
lokasi_contoh_error_v5 = direktori_outputs / "contoh_error_model_terbaik_v5.csv"

ringkasan_error.to_csv(lokasi_ringkasan_error_v5, index=False, encoding="utf-8")
contoh_error.to_csv(lokasi_contoh_error_v5, index=False, encoding="utf-8")

print("Error analysis selesai.")
display(ringkasan_error)
display(contoh_error.head(20))

Error analysis selesai.


,dataset_name,jenis_error,jumlah_data
0,DeepURLBench,benar,53709
1,DeepURLBench,false_negative,1433
2,DeepURLBench,false_positive,16537
3,PhiUSIIL,benar,45446
4,PhiUSIIL,false_negative,539
5,PhiUSIIL,false_positive,1354
6,PhreshPhish,benar,47026
7,PhreshPhish,false_negative,2248
8,PhreshPhish,false_positive,9899
9,Tranco,benar,19197


,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,y_true,y_pred,probabilitas_berisiko,jenis_error
105156,https://gofundchildren.com/,gofundchildren.com,phish,1,PhreshPhish,huggingface_phreshphish,test,1,0,0.149997,false_negative
26427,https://put.chftrials.com/ap/signin,put.chftrials.com,0,1,PhiUSIIL,local_phiusiil_raw,original,1,0,0.149909,false_negative
83994,https://lite.evernote.com/note/9921e238-b9a4-d...,lite.evernote.com,phish,1,PhreshPhish,huggingface_phreshphish,train,1,0,0.149906,false_negative
58939,https://gomezcotta.com/index/swicom/,gomezcotta.com,phish,1,PhreshPhish,huggingface_phreshphish,train,1,0,0.149901,false_negative
108910,https://www.dambos.com/gdocument,dambos.com,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,1,0,0.149744,false_negative
63846,https://forum.containerize.com/t/complete-list...,forum.containerize.com,phish,1,PhreshPhish,huggingface_phreshphish,train,1,0,0.149743,false_negative
92131,https://www.joinbestgroups.com/uadmin/plugins/...,joinbestgroups.com,phish,1,PhreshPhish,huggingface_phreshphish,test,1,0,0.149730,false_negative
115763,https://puttingkey.com/americafirst.com_mobile,puttingkey.com,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,1,0,0.149596,false_negative
164704,https://www.fullversionforever.com/tag/slime-r...,fullversionforever.com,malware,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,1,0,0.149568,false_negative
95058,https://joinbestgroups.com/uadmin/plugins/core...,joinbestgroups.com,phish,1,PhreshPhish,huggingface_phreshphish,test,1,0,0.149568,false_negative


## Validasi File Output

In [13]:
file_output_v5 = [
    lokasi_model_rf_v5,
    lokasi_model_xgb_v5,
    lokasi_model_terbaik_v5,
    lokasi_evaluasi_v5,
    lokasi_laporan_terbaik,
    lokasi_confusion_v5,
    lokasi_threshold_v5,
    lokasi_threshold_json_v5,
    lokasi_final_threshold,
    lokasi_confusion_final,
    lokasi_importance_v5,
    lokasi_ringkasan_error_v5,
    lokasi_contoh_error_v5,
    lokasi_split_v5,
]

data_validasi_output_v5 = []

for lokasi in file_output_v5:
    lokasi = Path(lokasi)
    data_validasi_output_v5.append({
        "nama_file": lokasi.name,
        "tersedia": lokasi.exists(),
        "ukuran_mb": round(lokasi.stat().st_size / (1024 * 1024), 2) if lokasi.exists() else 0,
        "lokasi": str(lokasi),
    })

data_validasi_output_v5 = pd.DataFrame(data_validasi_output_v5)

lokasi_validasi_output_v5 = direktori_outputs / "validasi_training_model_multi_dataset_v5.csv"
data_validasi_output_v5.to_csv(lokasi_validasi_output_v5, index=False, encoding="utf-8")

print("Validasi file output selesai.")
display(data_validasi_output_v5)

Validasi file output selesai.


,nama_file,tersedia,ukuran_mb,lokasi
0,model_rf_multi_dataset_v5.pkl,True,584.68,C:\Users\ASUS\PHISHING\models\model_rf_multi_d...
1,model_xgb_multi_dataset_v5.pkl,True,1.89,C:\Users\ASUS\PHISHING\models\model_xgb_multi_...
2,model_terbaik_multi_dataset_v5.pkl,True,584.68,C:\Users\ASUS\PHISHING\models\model_terbaik_mu...
3,hasil_evaluasi_model_multi_dataset_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\hasil_e...
4,classification_report_model_terbaik_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\classif...
5,confusion_matrix_model_terbaik_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\confusi...
6,threshold_tuning_model_terbaik_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\thresho...
7,threshold_model_terbaik_v5.json,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\thresho...
8,hasil_final_threshold_model_terbaik_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\hasil_f...
9,confusion_matrix_threshold_model_terbaik_v5.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\confusi...


## Metadata Final Training

In [14]:
metadata_training_v5 = {
    "nama_notebook": "15_training_model_multi_dataset_v5.ipynb",
    "nama_tahap": "Training Model Multi Dataset V5",
    "status": "selesai",
    "model_terbaik": model_terbaik_nama,
    "threshold_terbaik": threshold_terbaik,
    "jumlah_data": int(len(X)),
    "jumlah_fitur": int(len(daftar_fitur_v5)),
    "jumlah_train": int(len(X_train)),
    "jumlah_test": int(len(X_test)),
    "fitur": daftar_fitur_v5,
    "hasil_evaluasi": data_evaluasi_v5.to_dict(orient="records"),
    "hasil_final_threshold": data_final_threshold.to_dict(orient="records")[0],
    "file_input": {
        "fitur_training_v5": str(lokasi_fitur_v5),
        "target_training_v5": str(lokasi_target_v5),
        "dataset_training_v5": str(lokasi_dataset_v5),
        "daftar_fitur_v5": str(lokasi_daftar_fitur_v5),
    },
    "file_output": {
        "model_rf_v5": str(lokasi_model_rf_v5),
        "model_xgb_v5": str(lokasi_model_xgb_v5),
        "model_terbaik_v5": str(lokasi_model_terbaik_v5),
        "evaluasi_v5": str(lokasi_evaluasi_v5),
        "threshold_json": str(lokasi_threshold_json_v5),
        "feature_importance": str(lokasi_importance_v5),
        "validasi_output": str(lokasi_validasi_output_v5),
    },
    "catatan": [
        "Split train/test menggunakan GroupShuffleSplit berdasarkan domain.",
        "Sample weight memakai class balancing dan dataset source balancing.",
        "Target 1 berarti URL berisiko, termasuk phishing dan malware.",
        "Model terbaik dipilih berdasarkan roc_auc, f1 berisiko, dan recall berisiko.",
        "File model dan dataset besar jangan dipush ke GitHub biasa.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata_training_v5 = direktori_outputs / "metadata_training_model_multi_dataset_v5.json"

with open(lokasi_metadata_training_v5, "w", encoding="utf-8") as file:
    json.dump(metadata_training_v5, file, indent=4, ensure_ascii=False)

catatan_training_v5 = f"""
CATATAN FINAL TRAINING MODEL MULTI DATASET V5

Notebook:
15_training_model_multi_dataset_v5.ipynb

Model terbaik:
{model_terbaik_nama}

Threshold terbaik:
{threshold_terbaik}

Jumlah data:
{len(X)}

Jumlah fitur:
{len(daftar_fitur_v5)}

File model terbaik:
{lokasi_model_terbaik_v5}

Tahap berikutnya:
16_integrasi_engine_v5.ipynb

Catatan:
Model V5 dilatih dari PhiUSIIL, PhreshPhish, DeepURLBench, dan Tranco.
Split dibuat berdasarkan domain untuk mengurangi kebocoran pola URL yang terlalu mirip.
"""

lokasi_catatan_training_v5 = direktori_outputs / "catatan_final_training_model_multi_dataset_v5.txt"
lokasi_catatan_training_v5.write_text(catatan_training_v5, encoding="utf-8")

print(catatan_training_v5)
print("Metadata disimpan:", lokasi_metadata_training_v5)
print("Catatan disimpan:", lokasi_catatan_training_v5)


CATATAN FINAL TRAINING MODEL MULTI DATASET V5

Notebook:
15_training_model_multi_dataset_v5.ipynb

Model terbaik:
Random Forest Multi Dataset V5

Threshold terbaik:
0.15

Jumlah data:
975090

Jumlah fitur:
49

File model terbaik:
C:\Users\ASUS\PHISHING\models\model_terbaik_multi_dataset_v5.pkl

Tahap berikutnya:
16_integrasi_engine_v5.ipynb

Catatan:
Model V5 dilatih dari PhiUSIIL, PhreshPhish, DeepURLBench, dan Tranco.
Split dibuat berdasarkan domain untuk mengurangi kebocoran pola URL yang terlalu mirip.

Metadata disimpan: C:\Users\ASUS\PHISHING\reports\outputs\metadata_training_model_multi_dataset_v5.json
Catatan disimpan: C:\Users\ASUS\PHISHING\reports\outputs\catatan_final_training_model_multi_dataset_v5.txt
